# Official Zwift API Endpoint Tester

Use this notebook to manually test Zwift endpoints used by the project.

Tested endpoint groups:
- `/api/public/events/{eventId}` (event -> subgroup bridge)
- `/api/link/events/subgroups/{subgroupId}/segment-results`
- `/api/link/events/subgroups/{subgroupId}/live-data`
- `/api/link/racing-profile`
- `/api/link/power-curve/*`
- `/api/thirdparty/activity/{activityId}`

## Notes
- This notebook now supports resolving subgroup IDs from event IDs using `/api/public/events/{eventId}`.
- `/api/public/events/{eventId}` is useful in practice but is not listed in Zwift developer docs.

## Before you run
1. Ensure your `.env` has `ZWIFT_CLIENT_ID` and `ZWIFT_CLIENT_SECRET`.
2. For user-scoped endpoints, provide `USER_ACCESS_TOKEN` (from OAuth authorization code flow).
3. Provide one of these for subgroup endpoints:
   - `SUBGROUP_ID` directly, or
   - `EVENT_IDS` (resolve from `/api/public/events/{eventId}`), or
   - `ACTIVITY_ID` (resolve from `/api/thirdparty/activity/{activityId}`).

In [6]:
import json
import os
import requests
from dotenv import load_dotenv

load_dotenv()

ZWIFT_AUTH_BASE = os.getenv("ZWIFT_AUTH_BASE_URL", "https://secure.zwift.com/auth/realms/zwift").rstrip("/")
ZWIFT_API_BASE = os.getenv("ZWIFT_API_BASE_URL", "https://us-or-rly101.zwift.com").rstrip("/")
ZWIFT_CLIENT_ID = (os.getenv("ZWIFT_CLIENT_ID") or "").strip()
ZWIFT_CLIENT_SECRET = (os.getenv("ZWIFT_CLIENT_SECRET") or "").strip()

print("ZWIFT_API_BASE:", ZWIFT_API_BASE)
print("ZWIFT_AUTH_BASE:", ZWIFT_AUTH_BASE)
print("ZWIFT_CLIENT_ID set:", bool(ZWIFT_CLIENT_ID))
print("ZWIFT_CLIENT_SECRET set:", bool(ZWIFT_CLIENT_SECRET))

ZWIFT_API_BASE: https://us-or-rly101.zwift.com
ZWIFT_AUTH_BASE: https://secure.zwift.com/auth/realms/zwift
ZWIFT_CLIENT_ID set: True
ZWIFT_CLIENT_SECRET set: True


In [7]:
# Fill these values before running endpoint tests
EVENT_IDS = [5370578]  # e.g. [5370578]

# Provide one of these to run subgroup endpoints:
# 1) SUBGROUP_ID directly, OR
# 2) EVENT_IDS (resolve via /api/public/events/{eventId}), OR
# 3) ACTIVITY_ID (resolve via /api/thirdparty/activity/{activityId}).
SUBGROUP_ID = ""
ACTIVITY_ID = ""

# Needed for user-scoped endpoints and activity-based subgroup derivation.
USER_ACCESS_TOKEN = (os.getenv("ZWIFT_USER_ACCESS_TOKEN") or "").strip()

# Optional request tuning
TIMEOUT_SECONDS = 25

print("EVENT_IDS:", EVENT_IDS or "<set one or more event ids>")
print("SUBGROUP_ID (optional override):", SUBGROUP_ID or "<auto from EVENT_IDS / ACTIVITY_ID>")
print("ACTIVITY_ID (optional):", ACTIVITY_ID or "<set to derive subgroup from activity>")
print("USER_ACCESS_TOKEN set:", bool(USER_ACCESS_TOKEN))

EVENT_IDS: [5370578]
SUBGROUP_ID (optional override): <set or derive from ACTIVITY_ID>
ACTIVITY_ID (optional): <set to auto-derive subgroup>
USER_ACCESS_TOKEN set: False


In [8]:
def get_app_token() -> str:
    if not ZWIFT_CLIENT_ID or not ZWIFT_CLIENT_SECRET:
        raise ValueError("Missing ZWIFT_CLIENT_ID / ZWIFT_CLIENT_SECRET")

    token_url = f"{ZWIFT_AUTH_BASE}/protocol/openid-connect/token"
    resp = requests.post(
        token_url,
        data={
            "client_id": ZWIFT_CLIENT_ID,
            "client_secret": ZWIFT_CLIENT_SECRET,
            "grant_type": "client_credentials",
        },
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        timeout=TIMEOUT_SECONDS,
    )
    resp.raise_for_status()
    return resp.json()["access_token"]


def zwift_get(path: str, token: str, params: dict | None = None):
    url = f"{ZWIFT_API_BASE}{path}"
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json",
    }
    resp = requests.get(url, headers=headers, params=params or {}, timeout=TIMEOUT_SECONDS)
    try:
        body = resp.json()
    except Exception:
        body = {"_raw": resp.text}
    return resp.status_code, body


def show_response(name: str, status: int, body, max_list_items: int = 5):
    print(f"\n{name}")
    print(f"status: {status}")
    if isinstance(body, list) and len(body) > max_list_items:
        print(json.dumps(body[:max_list_items], indent=2, default=str))
        print(f"... ({len(body) - max_list_items} more items)")
    elif isinstance(body, dict) and "entries" in body and isinstance(body["entries"], list) and len(body["entries"]) > max_list_items:
        slim = dict(body)
        slim["entries"] = body["entries"][:max_list_items]
        print(json.dumps(slim, indent=2, default=str))
        print(f"... ({len(body['entries']) - max_list_items} more entries)")
    else:
        print(json.dumps(body, indent=2, default=str))

In [9]:
def test_segment_results(app_token: str, subgroup_id: str):
    if not subgroup_id:
        raise ValueError("Set SUBGROUP_ID first")
    status, body = zwift_get(f"/api/link/events/subgroups/{subgroup_id}/segment-results", app_token)
    show_response("segment-results", status, body)
    return status, body


def test_live_data(app_token: str, subgroup_id: str, include_avatar: bool = False, include_position: bool = False):
    if not subgroup_id:
        raise ValueError("Set SUBGROUP_ID first")
    status, body = zwift_get(
        f"/api/link/events/subgroups/{subgroup_id}/live-data",
        app_token,
        params={
            "page": 0,
            "limit": 10,
            "includeAvatar": str(include_avatar).lower(),
            "includePosition": str(include_position).lower(),
        },
    )
    show_response("live-data", status, body)
    return status, body


def test_racing_profile(user_token: str):
    if not user_token:
        raise ValueError("Set USER_ACCESS_TOKEN first")
    status, body = zwift_get(
        "/api/link/racing-profile",
        user_token,
        params={"includeCompetitionMetrics": "true"},
    )
    show_response("racing-profile", status, body)
    return status, body


def test_power_curve(user_token: str, days: int = 30, year: int | None = None, activity_id: str | None = None):
    if not user_token:
        raise ValueError("Set USER_ACCESS_TOKEN first")

    statuses = {}

    status, body = zwift_get("/api/link/power-curve/best/all-time", user_token)
    show_response("power-curve/best/all-time", status, body)
    statuses["best_all_time"] = status

    status, body = zwift_get("/api/link/power-curve/best/last", user_token, params={"days": days})
    show_response(f"power-curve/best/last?days={days}", status, body)
    statuses["best_last"] = status

    if year is not None:
        status, body = zwift_get(f"/api/link/power-curve/best/year/{year}", user_token)
        show_response(f"power-curve/best/year/{year}", status, body)
        statuses["best_year"] = status

    status, body = zwift_get("/api/link/power-curve/power-profile", user_token)
    show_response("power-curve/power-profile", status, body)
    statuses["power_profile"] = status

    if activity_id:
        status, body = zwift_get(f"/api/link/power-curve/activity/{activity_id}", user_token)
        show_response("power-curve/activity/{activityId}", status, body)
        statuses["power_curve_activity"] = status

    return statuses


def test_activity(user_token: str, activity_id: str):
    if not user_token:
        raise ValueError("Set USER_ACCESS_TOKEN first")
    if not activity_id:
        raise ValueError("Set ACTIVITY_ID first")
    status, body = zwift_get(f"/api/thirdparty/activity/{activity_id}", user_token)
    show_response("thirdparty/activity/{activityId}", status, body)
    return status, body


def resolve_subgroup_ids_from_activity(user_token: str, activity_id: str) -> list[str]:
    """
    Resolve subgroup id(s) using official endpoint:
      GET /api/thirdparty/activity/{activityId}

    Returns a unique list of subgroup IDs found in the response.
    """
    status, body = test_activity(user_token, activity_id)
    if status != 200 or not isinstance(body, dict):
        raise ValueError(f"Activity lookup failed with status={status}")

    subgroup_ids: list[str] = []

    direct_id = body.get("eventSubgroupId")
    if direct_id is not None and str(direct_id).strip():
        subgroup_ids.append(str(direct_id).strip())

    # Defensive fallback for any nested structures that include eventSubgroupId.
    for key in ("entries", "results", "segments"):
        val = body.get(key)
        if isinstance(val, list):
            for item in val:
                if isinstance(item, dict):
                    sid = item.get("eventSubgroupId")
                    sid = str(sid).strip() if sid is not None else ""
                    if sid and sid not in subgroup_ids:
                        subgroup_ids.append(sid)

    if not subgroup_ids:
        raise ValueError("No eventSubgroupId found in activity response")

    print("Resolved subgroup IDs from activity:", subgroup_ids)
    return subgroup_ids

In [10]:
# 1) Get app token (used for subgroup endpoints)
APP_TOKEN = get_app_token()
print("APP_TOKEN acquired:", bool(APP_TOKEN))

APP_TOKEN acquired: True


In [11]:
# 2) Test app-token endpoints
# Resolution order:
#   SUBGROUP_ID -> EVENT_IDS(public event endpoint) -> ACTIVITY_ID -> fail

def resolve_subgroup_ids_from_public_event(event_id: int | str) -> list[str]:
    url = f"{ZWIFT_API_BASE}/api/public/events/{event_id}"
    resp = requests.get(url, headers={"Accept": "application/json"}, timeout=TIMEOUT_SECONDS)
    try:
        body = resp.json()
    except Exception:
        body = {"_raw": resp.text}

    print(f"\npublic event lookup event_id={event_id} status={resp.status_code}")
    if resp.status_code != 200 or not isinstance(body, dict):
        return []

    ids: list[str] = []
    for subgroup in body.get("eventSubgroups", []) or []:
        sid = subgroup.get("id")
        sid_str = str(sid).strip() if sid is not None else ""
        if sid_str and sid_str not in ids:
            ids.append(sid_str)
    return ids


if SUBGROUP_ID:
    subgroup_ids = [SUBGROUP_ID]
else:
    subgroup_ids = []

    if EVENT_IDS:
        for eid in EVENT_IDS:
            subgroup_ids.extend(resolve_subgroup_ids_from_public_event(eid))
        subgroup_ids = sorted(set(subgroup_ids))

    if not subgroup_ids and ACTIVITY_ID:
        subgroup_ids = resolve_subgroup_ids_from_activity(USER_ACCESS_TOKEN, ACTIVITY_ID)

    if not subgroup_ids:
        raise ValueError(
            "Could not resolve subgroup IDs. Provide SUBGROUP_ID, or EVENT_IDS with /api/public/events support, "
            "or ACTIVITY_ID + USER_ACCESS_TOKEN."
        )

selected_subgroup_id = subgroup_ids[0]
print("Resolved subgroup_ids:", subgroup_ids)
print("Using subgroup_id:", selected_subgroup_id)

segment_status, segment_body = test_segment_results(APP_TOKEN, selected_subgroup_id)
live_status, live_body = test_live_data(APP_TOKEN, selected_subgroup_id, include_avatar=True, include_position=False)

print("\napp-token endpoint status summary:")
print({
    "segment_results": segment_status,
    "live_data": live_status,
})

ValueError: Cannot derive subgroup IDs from EVENT_IDS using only official endpoints. Provide SUBGROUP_ID directly, or provide ACTIVITY_ID + USER_ACCESS_TOKEN.

In [ ]:
# 3) Test user-token endpoints (requires USER_ACCESS_TOKEN)
profile_status, profile_body = test_racing_profile(USER_ACCESS_TOKEN)
power_statuses = test_power_curve(USER_ACCESS_TOKEN, days=30, year=2026, activity_id=ACTIVITY_ID or None)

print("\nuser-token endpoint status summary:")
print({
    "racing_profile": profile_status,
    **power_statuses,
})

In [ ]:
# 4) Optional: print activity details directly (requires ACTIVITY_ID)
if ACTIVITY_ID:
    activity_status, activity_body = test_activity(USER_ACCESS_TOKEN, ACTIVITY_ID)
    print("activity status:", activity_status)
else:
    print("Set ACTIVITY_ID to run /api/thirdparty/activity/{activityId}")